# Parameter Correctness: Did the Agent Pass What the User Asked?

Based on: [Lost in Execution: Multilingual Robustness of Tool Calling](https://arxiv.org/abs/2601.05366) (Jan 2026)

## The Problem

The agent picks the right tool and the parameters pass constraint validation. But do they match what the user actually asked?

```
User: "Find flights from New York to London"
Agent: search_flights(origin="JFK", destination="LHR", date="2026-03-28")
```

Is `origin="JFK"` correct? The user said "New York", not "JFK". Is `date="2026-03-28"` correct? The user did not specify a date.

The [Lost in Execution paper](https://arxiv.org/abs/2601.05366) found that **parameter value language mismatch** is the dominant failure mode in multilingual tool calling.

## What We Compare

| Check | What It Catches | Cost |
|-------|----------------|:----:|
| Deterministic constraints (Demo 02) | Format, range, required | Free |
| **Semantic parameter check (this demo)** | **Intent mismatch, assumed values** | **1 LLM call** |

The semantic check asks an LLM: "Given the user query, are these parameters correct?"

In [ ]:
# %pip install strands-agents strands-agents-evals boto3

## Step 1: Simulated tool calls with known issues

**What this does:** Defines 5 tool calls with pre-labeled correctness, including cases with subtle intent mismatches.

**The difference between valid parameters and correct parameters:** Demo 02 (Constraint Validation) checks whether parameters are *valid* — proper format, in range, non-empty. This demo checks whether parameters are *correct* — whether they accurately reflect what the user actually asked for. A parameter can be valid but incorrect:

| Example | Valid? | Correct? | Why |
|---------|:------:|:--------:|-----|
| `origin="NYC"` when user said "NYC" | Yes | Yes | Exact match to user input |
| `origin="JFK"` when user said "New York" | Yes | Yes | Reasonable mapping (JFK is a New York airport) |
| `date="2026-06-15"` when user said nothing about dates | Yes | **No** | Valid date format, but the user never specified a date — the agent assumed it |
| `city="London"` when user said "Paris" | Yes | **No** | Valid city name, but contradicts the user's request |

This distinction matters because constraint validation (free, deterministic) catches the first kind of error, but only semantic evaluation (LLM-based) catches the second.

> **What to look for:** The 5 test cases include 3 correct mappings and 2 incorrect ones. The incorrect cases are: (1) an assumed date the user never mentioned, and (2) a wrong destination city. Both pass constraint validation but fail semantic correctness.

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Fix for Jupyter async event loop

from strands_evals import Experiment, Case
from strands_evals.evaluators import OutputEvaluator

MODEL = "gpt-4o-mini"

# Each entry: (user_query, tool_name, params, expected_correct)
TOOL_CALLS = [
    {
        "name": "correct_mapping",
        "query": "Find flights from NYC to London for next Friday",
        "tool": "search_flights",
        "params": {"origin": "NYC", "destination": "London", "date": "2026-03-27"},
        "expected_correct": True,
    },
    {
        "name": "airport_code_mismatch",
        "query": "Find flights from New York to London",
        "tool": "search_flights",
        "params": {"origin": "JFK", "destination": "LHR", "date": "2026-03-28"},
        "expected_correct": True,  # JFK/LHR are valid mappings
    },
    {
        "name": "assumed_date",
        "query": "Find flights from NYC to London",
        "tool": "search_flights",
        "params": {"origin": "NYC", "destination": "London", "date": "2026-06-15"},
        "expected_correct": False,  # User did not mention a date
    },
    {
        "name": "wrong_destination",
        "query": "Search hotels in Paris for March 20-22",
        "tool": "search_hotels",
        "params": {"city": "London", "check_in": "2026-03-20", "check_out": "2026-03-22"},
        "expected_correct": False,  # User said Paris, agent passed London
    },
    {
        "name": "correct_conversion",
        "query": "Convert 500 USD to EUR",
        "tool": "get_currency_exchange",
        "params": {"from_currency": "USD", "to_currency": "EUR", "amount": 500},
        "expected_correct": True,
    },
]

print(f"📋 {len(TOOL_CALLS)} tool calls to evaluate\n")
for tc in TOOL_CALLS:
    icon = "✅" if tc["expected_correct"] else "❌"
    print(f"  {icon} {tc['name']}: {tc['tool']}({tc['params']})")
    print(f"     User asked: '{tc['query']}'")

---
## Step 2: Semantic parameter evaluation with OutputEvaluator

**What this does:** Uses an LLM judge with a rubric to score whether each tool call's parameters match the user's intent, not merely whether the parameters are structurally valid.

**Why semantic evaluation is needed:** Deterministic constraint checks (Demo 02) cannot assess intent. They verify that `date="2026-06-15"` is a valid future date, but they cannot know that the user never asked for June 15th. Only an LLM judge can compare the parameters to the original query and determine whether the agent made up information or misunderstood the request.

> **What to look for:** The correct mappings ("correct_mapping", "airport_code_mismatch", "correct_conversion") should score 0.7-1.0. The "assumed_date" case should score lower (0.3-0.6) because the agent invented a date. The "wrong_destination" case should score near 0.0 because it directly contradicts the user's request. If the LLM judge scores "wrong_destination" highly, the rubric needs to be more explicit about penalizing contradictions.

In [ ]:
PARAM_RUBRIC = (
    "Score 0-1 whether the tool parameters match the user's intent.\n"
    "1.0: All parameters accurately reflect what the user asked\n"
    "0.7-0.9: Parameters are reasonable mappings (city name → airport code)\n"
    "0.3-0.6: Some parameters are assumed without user input (dates not mentioned)\n"
    "0.0-0.2: Parameters contradict the user's request (wrong city, wrong amount)"
)

evaluator = OutputEvaluator(rubric=PARAM_RUBRIC, model=MODEL)

cases = [
    Case(
        name=tc["name"],
        input=f"User query: {tc['query']}",
        expected_output=f"Tool: {tc['tool']}, Expected params based on query",
    )
    for tc in TOOL_CALLS
]

def param_task(case):
    for tc in TOOL_CALLS:
        if tc["name"] == case.name:
            return f"Tool called: {tc['tool']}({tc['params']})"
    return ""

print("=" * 70)
print("SEMANTIC PARAMETER EVALUATION")
print("=" * 70)

exp = Experiment(cases=cases, evaluators=[evaluator])
reports = exp.run_evaluations(param_task)
reports[0].display()

## Series Summary: 3 Layers of Tool Use Evaluation

```
Layer 1: Tool Selection (Demo 01)
   "Did the agent pick the right tool?"
   → ToolCalled (free) or TrajectoryEvaluator (LLM)

Layer 2: Constraint Validation (Demo 02)
   "Are the parameters valid?"
   → Deterministic rules: dates, ranges, required fields (free)

Layer 3: Parameter Correctness (Demo 03)
   "Do the parameters match user intent?"
   → OutputEvaluator with semantic rubric (LLM)
```

| Layer | What It Catches | Cost | Run When |
|-------|----------------|:----:|----------|
| 1. Tool selection | Wrong tool called | Free or 1 call | Every invocation |
| 2. Constraints | Invalid parameters | Free | Every tool call |
| 3. Semantic params | Intent mismatch | 1 call | Periodic / high-stakes |

**Use all 3 together:** Layer 1 and 2 are free and run on every call. Layer 3 runs on a sample or during quality reviews.